# 第十课：NumPy 迭代数组

本课依据菜鸟教程的 [NumPy 迭代数组](https://www.runoob.com/numpy/numpy-terating-over-array.html) 整理，学习普通循环、`flat` 和 `np.nditer()`。

NumPy 擅长矢量化运算。只有确实需要逐项处理、控制内存顺序或同时遍历多个数组时，才优先考虑迭代器。

In [1]:
import numpy as np
print("NumPy 版本：", np.__version__)

NumPy 版本： 2.5.3


## 1. 普通 `for` 循环

直接遍历二维数组时，每次得到第 0 轴上的一个子数组，也就是一行，而不是单个元素。

In [2]:
a = np.arange(6).reshape(2, 3)
print(a)
for row in a:
    print("一行：", row)

[[0 1 2]
 [3 4 5]]
一行： [0 1 2]
一行： [3 4 5]


## 2. `flat`：逐个元素迭代

`array.flat` 是扁平迭代器，按 C 顺序逐个访问元素。它不是新数组。

In [3]:
print("逐个元素：", end="")
for value in a.flat:
    print(value, end=" " )
print()
print("flat[4]：", a.flat[4])

逐个元素：0 1 2 3 4 5 
flat[4]： 4


## 3. `np.nditer()` 基础

`nditer` 是更灵活的数组迭代器。默认会根据数组的内存布局选择高效访问顺序，迭代变量通常是零维数组。

In [4]:
a = np.arange(6).reshape(2, 3)
for value in np.nditer(a):
    print(value.item(), end=" " )
print()

0 1 2 3 4 5 


## 4. 控制遍历顺序

- `order='C'`：最后一个轴变化最快，通常可理解为逐行
- `order='F'`：第一个轴变化最快，通常可理解为逐列
- `order='K'`：尽量按内存实际布局访问

In [5]:
m = np.arange(12).reshape(3, 4)
c_order = [x.item() for x in np.nditer(m, order="C")]
f_order = [x.item() for x in np.nditer(m, order="F")]
print("原数组：\n", m)
print("C 顺序：", c_order)
print("F 顺序：", f_order)

原数组：
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
C 顺序： [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
F 顺序： [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]


转置通常只是改变视图和步长，不会重新排列底层数据。因此 `order='K'` 遍历转置数组时，顺序可能与屏幕上逐行阅读不同。

In [6]:
transposed = m.T
print("转置数组：\n", transposed)
print("按 C 顺序：", [x.item() for x in np.nditer(transposed, order="C")])
print("按内存顺序：", [x.item() for x in np.nditer(transposed, order="K")])

转置数组：
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
按 C 顺序： [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]
按内存顺序： [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## 5. 在迭代中修改元素

默认操作数是只读的。要修改原数组，必须设置 `op_flags=['readwrite']`，并使用 `x[...]` 写回。实际计算中通常更推荐矢量化写法 `a *= 2`。

In [7]:
values = np.arange(6).reshape(2, 3)
with np.nditer(values, op_flags=["readwrite"]) as iterator:
    for x in iterator:
        x[...] = x * 2
print(values)
print("矢量化写法会更简洁：values *= 2")

[[ 0  2  4]
 [ 6  8 10]]
矢量化写法会更简洁：values *= 2


## 6. 同时取得多维坐标

`flags=['multi_index']` 让迭代器报告当前元素的多维索引。

In [9]:
grid = np.array([[10, 20, 30], [40, 50, 60]])
print("原数组：\n", grid)
with np.nditer(grid, flags=["multi_index"]) as iterator:
    for x in iterator:
        print(f"坐标 {iterator.multi_index} → {x.item()}")

原数组：
 [[10 20 30]
 [40 50 60]]
坐标 (0, 0) → 10
坐标 (0, 1) → 20
坐标 (0, 2) → 30
坐标 (1, 0) → 40
坐标 (1, 1) → 50
坐标 (1, 2) → 60


## 7. `external_loop`：成块返回数据

使用 `flags=['external_loop']` 时，每次得到一维数据块而非单个零维元素，可减少 Python 循环次数。块大小受内存布局和遍历顺序影响。

In [14]:
grid = np.arange(12).reshape(3, 4)
print("原数组：\n", grid)
print("F 顺序的数据块：")
for block in np.nditer(grid, flags=["external_loop"], order="F"):
    print(block)

原数组：
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
F 顺序的数据块：
[0 4 8]
[1 5 9]
[ 2  6 10]
[ 3  7 11]


## 8. 同时迭代可广播数组

`nditer([a, b])` 可以按广播规则同时提供对应元素。

In [16]:
left = np.arange(0, 60, 5).reshape(3, 4)
right = np.array([1, 2, 3, 4])
print("左数组：\n", left)
print("右数组：\n", right)  
pairs = []
for x, y in np.nditer([left, right]):
    pairs.append((x.item(), y.item()))
print("前 8 对：", pairs[:8])
print("广播结果：\n", left + right)

左数组：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
右数组：
 [1 2 3 4]
前 8 对： [(0, 1), (5, 2), (10, 3), (15, 4), (20, 1), (25, 2), (30, 3), (35, 4)]
广播结果：
 [[ 1  7 13 19]
 [21 27 33 39]
 [41 47 53 59]]


## 9. 优先矢量化

逐元素循环通常比 NumPy 的矢量化运算慢，也更冗长。能直接写数组表达式时，应优先使用数组表达式。`np.vectorize()` 主要是代码便利工具，通常不会带来真正的底层矢量化性能。

In [17]:
numbers = np.arange(1, 7)
loop_result = np.array([x * x + 1 for x in numbers])
numpy_result = numbers ** 2 + 1
print(loop_result)
print(numpy_result)
print("结果相同：", np.array_equal(loop_result, numpy_result))

[ 2  5 10 17 26 37]
[ 2  5 10 17 26 37]
结果相同： True


## 本课小结

- 普通循环遍历第 0 轴；`flat` 逐个遍历元素。
- `nditer` 支持 C/F/K 顺序、读写模式、坐标追踪和外部循环。
- 多个可广播数组可以一起迭代。
- 修改时使用 `op_flags=['readwrite']` 和 `x[...]`。
- 能矢量化时，优先使用 NumPy 数组运算。

## 本节练习

1. 用 `nditer` 按 F 顺序输出 `practice`。
2. 用 `multi_index` 找出值大于 6 的坐标和值。
3. 在副本中使用 `readwrite` 把所有奇数乘以 10。
4. 再用一行矢量化代码完成同样修改，并验证结果相同。

In [45]:
practice = np.arange(1, 10).reshape(3, 3)
print("原数组：\n", practice)
practice_ = np.arange(1, 10).reshape(3, 3)
# 请在这里完成练习
for i in np.nditer(practice,order='F'):
    print(i,end=' ')
print()

with np.nditer(practice, flags=['multi_index'], order='F') as it:
    for x in it:
        if x>6:
            print(f"坐标 {it.multi_index} → {x.item()}")

with np.nditer(practice, op_flags=['readwrite']) as it:
    for x in it:
        if x%2 == 1:
            x[...] = x * 10
print("修改后的数组：\n", practice)
values = practice_ % 2 ==1

practice_[values] = practice_[values] * 10
print("修改后的数组：\n", practice_)

原数组：
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
1 4 7 2 5 8 3 6 9 
坐标 (2, 0) → 7
坐标 (2, 1) → 8
坐标 (2, 2) → 9
修改后的数组：
 [[10  2 30]
 [ 4 50  6]
 [70  8 90]]
修改后的数组：
 [[10  2 30]
 [ 4 50  6]
 [70  8 90]]


## 参考答案

In [36]:
print("F 顺序：", [x.item() for x in np.nditer(practice, order="F")])
with np.nditer(practice, flags=["multi_index"]) as it:
    for x in it:
        if x > 6:
            print("大于 6：", it.multi_index, x.item())
iter_result = practice.copy()
with np.nditer(iter_result, op_flags=["readwrite"]) as it:
    for x in it:
        if x % 2 == 1:
            x[...] = x * 10
vector_result = practice.copy()
vector_result[vector_result % 2 == 1] *= 10
print(iter_result)
print("与矢量化结果相同：", np.array_equal(iter_result, vector_result))

F 顺序： [1, 4, 7, 2, 5, 8, 3, 6, 9]
大于 6： (2, 0) 7
大于 6： (2, 1) 8
大于 6： (2, 2) 9
[[10  2 30]
 [ 4 50  6]
 [70  8 90]]
与矢量化结果相同： True


---

下一课：[NumPy 数组操作](https://www.runoob.com/numpy/numpy-array-manipulation.html)。